# Aya / Cohere algorithm smoke tests -- TRL backend

Runs every AlignTune algorithm against an Aya/Cohere model (`cohere`,
`cohere2`, `aya_vision` or `cohere_compass`) on the **TRL** backend: SFT,
12 RL / preference algorithms, RAFT, and distillation (Standard and SDFT).

**How to use this notebook:** pick a model in the cell below section 0, then
run every cell top to bottom once; re-run individual algorithm cells as
needed. To swap models, edit *only* that one cell and re-run from there down.
The validation cell in section 0 checks that every algorithm used here is
registered for `backend="trl"` before any GPU time is spent.

**Requirements:** a GPU (the 8B Aya models need real VRAM) and access to the
gated model repos. Every cell runs a short smoke test (`MAX_STEPS`); it checks
that each algorithm trains end to end, not that it converges. The
vision-language models (`aya_vision`, `cohere_compass`) are trained on text
data only, with LoRA scoped to the language model.


## 0. Setup

In [ ]:
# Tiny Aya (and possibly other checkpoints above) are gated on the Hub:
# accept the licence on the model page first, then export HF_TOKEN in your
# shell (or paste it at the login prompt below).
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # set to the GPU you want to use
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()  # prompts for a token interactively


### Model selection -- edit this one cell to switch models

In [ ]:
# -- Pick ONE model -- comment out the rest ---------------------------------
# All four families are supported: Aya
# Expanse / Tiny Aya load as plain causal LMs; Aya Vision / North Micro Vision
# load via AutoModelForImageTextToText, with LoRA scoped to the language
# model only -- this notebook trains on TEXT data only.

MODEL_NAME = "CohereLabs/aya-expanse-8b"        # cohere      -- Aya Expanse (8B, causal LM)
# MODEL_NAME = "CohereLabs/aya-vision-8b"       # aya_vision  -- Aya Vision (8B, vision-language; text-only here)

# Tiny Aya (cohere2) and North Micro Vision (cohere_compass) are the other two
# supported families. Fill in the North Micro Vision checkpoint you have access to.
# MODEL_NAME = "CohereLabs/tiny-aya-base"                       # cohere2
# MODEL_NAME = "<fill in the exact North Micro Vision repo id>"  # cohere_compass

print("Testing with:", MODEL_NAME)

# Smoke-test length for every algorithm cell; raise for a real training run.
MAX_STEPS = 2

### Pre-flight: confirm every algorithm below is registered for this backend

In [ ]:
# Sanity check BEFORE spending GPU time: confirm every algorithm below is
# actually registered for backend="trl" in this install.
from aligntune.core.backend_factory import BackendFactory, TrainingType, BackendType, RLAlgorithm

BACKEND = BackendType.TRL
RL_ALGOS = ["dpo", "online_dpo", "ppo", "grpo", "gbmpo", "counterfact_grpo",
            "pace", "gspo", "dapo", "drgrpo", "spin", "orpo"]

def registered(training_type, algorithm=None):
    return (training_type, BACKEND, algorithm) in BackendFactory._backends

missing = []
if not registered(TrainingType.SFT):
    missing.append("sft")
for a in RL_ALGOS:
    if not registered(TrainingType.RL, RLAlgorithm(a)):
        missing.append(a)
DISTILL_METHODS = ["distillation", "sdft"]
for m in DISTILL_METHODS:
    if not registered(TrainingType.DISTILL, m):
        missing.append(m)
# RAFT is not routed through BackendFactory._backends (create_raft_trainer
# dispatches on backend= directly) so it isn't checked here.

print(f"{len(RL_ALGOS) + 1 + len(DISTILL_METHODS) - len(missing)}/{len(RL_ALGOS) + 1 + len(DISTILL_METHODS)} registered for backend='trl'")
if missing:
    print("NOT registered:", missing)
assert not missing, f"missing backend registrations: {missing}"

## 1. SFT

In [ ]:
from aligntune.core.backend_factory import create_sft_trainer

trainer = create_sft_trainer(
    model_name=MODEL_NAME,
    dataset_name="tatsu-lab/alpaca",
    backend="trl",
    output_dir="./out_sft",
    num_epochs=1,
    max_steps = MAX_STEPS,
    batch_size=2,
    learning_rate=2e-4,
    max_seq_length=256,
    max_samples=16,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
)

results = trainer.train()
print("SFT training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 2. DPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="Anthropic/hh-rlhf",
    algorithm="dpo",
    backend="trl",
    output_dir="./out_dpo",
    max_steps=MAX_STEPS,
    batch_size=1,
    learning_rate=5e-6,
    max_seq_length=256,
    max_samples=16,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
)

results = trainer.train()
print("DPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 3. Online-DPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

# Online DPO requires an explicit reward source -- it does not default to
# reusing the policy model as its own (untrained) reward model.
trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="Anthropic/hh-rlhf",
    algorithm="online_dpo",
    backend="trl",
    output_dir="./out_online_dpo",
    max_steps=MAX_STEPS,
    batch_size=1,
    learning_rate=5e-6,
    max_seq_length=256,
    max_samples=16,
    reward_model_name=MODEL_NAME,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
)

results = trainer.train()
print("Online DPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 4. PPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name="CohereLabs/tiny-aya-base",
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt"},
    algorithm="ppo",
    backend="trl",
    output_dir="./out_ppo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    learning_rate=1e-5,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    response_length=64,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("PPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 5. GRPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="grpo",
    backend="trl",
    output_dir="./out_grpo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("GRPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 6. GBMPO

Unified config, 4 divergence types (L2/L2KL/ProbL2/ProbL2KL) -- default divergence used here; pass `gbmpo_divergence_type=` to pick one explicitly.

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="gbmpo",
    backend="trl",
    output_dir="./out_gbmpo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("GBMPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 7. Counterfactual GRPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="counterfact_grpo",
    backend="trl",
    output_dir="./out_counterfact_grpo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("Counterfactual GRPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 8. PACE

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="pace",
    backend="trl",
    output_dir="./out_pace",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("PACE training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 9. GSPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="gspo",
    backend="trl",
    output_dir="./out_gspo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("GSPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 10. DAPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="dapo",
    backend="trl",
    output_dir="./out_dapo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("DAPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 11. Dr. GRPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="openai/gsm8k",
    config_name="main",
    split="train",
    column_mapping={"question": "prompt", "answer": "response"},
    algorithm="drgrpo",
    backend="trl",
    output_dir="./out_drgrpo",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    num_generations=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    max_completion_length=48,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    rewards=[{"type": "math_correctness", "weight": 1.0, "params": {}}],
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("Dr. GRPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 12. SPIN

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="tatsu-lab/alpaca",
    split="train",
    algorithm="spin",
    backend="trl",
    output_dir="./out_spin",
    num_epochs=1,
    max_steps=MAX_STEPS,
    batch_size=2,
    learning_rate=1e-4,
    max_seq_length=256,
    max_samples=16,
    max_prompt_length=128,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    num_rounds=1,
    generation_max_length=128,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=42,
)

results = trainer.train()
print("SPIN training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 13. ORPO

In [ ]:
from aligntune.core.backend_factory import create_rl_trainer

trainer = create_rl_trainer(
    model_name=MODEL_NAME,
    dataset_name="Anthropic/hh-rlhf",
    algorithm="orpo",
    backend="trl",
    output_dir="./out_orpo",
    max_steps=MAX_STEPS,
    batch_size=1,
    learning_rate=5e-6,
    max_seq_length=256,
    max_samples=16,
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
)

results = trainer.train()
print("ORPO training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 14. RAFT

`create_raft_trainer` always loads the **full** model (no LoRA support -- passing `lora_r`/`use_peft` here would raise a `TypeError`, they fall straight through to `TrainingArguments`). With an 8B Aya checkpoint this is a full fine-tune and needs real GPU memory; drop to a smaller checkpoint if you're VRAM constrained.

In [ ]:
from aligntune.core.backend_factory import create_raft_trainer

train_examples = [
    {
        "question": "What is the capital of France?",
        "answer": "The capital of France is Paris.",
        "golden_docs": [{"title": "France", "text": "France is a country in Western Europe. Its capital is Paris."}],
        "distractor_docs": [{"title": "Germany", "text": "Germany is a country in Central Europe. Its capital is Berlin."}],
    },
    {
        "question": "Who wrote Romeo and Juliet?",
        "answer": "Romeo and Juliet was written by William Shakespeare.",
        "golden_docs": [{"title": "Shakespeare", "text": "William Shakespeare was an English playwright who wrote Romeo and Juliet."}],
        "distractor_docs": [{"title": "Dickens", "text": "Charles Dickens was an English novelist who wrote Oliver Twist."}],
    },
    {
        "question": "What is the boiling point of water at sea level?",
        "answer": "Water boils at 100 degrees Celsius at sea level.",
        "golden_docs": [{"title": "Water", "text": "At sea level, water boils at 100 degrees Celsius (212 Fahrenheit)."}],
        "distractor_docs": [{"title": "Ice", "text": "Water freezes at 0 degrees Celsius at standard atmospheric pressure."}],
    },
]

trainer = create_raft_trainer(
     model_name="CohereLabs/tiny-aya-base",
    train_examples=train_examples,
    backend="trl",
    output_dir="./out_raft",
    num_epochs=1,
    batch_size=2,
    learning_rate=2e-4,
    warmup_steps=0,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    max_steps=2,
    report_to=[],
)

trainer.train()
print("RAFT training completed.")

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## 15. Distillation (2 methods)

`create_distill_trainer` picks the method from the kwargs: `teacher_model_kind` -> SDFT; otherwise Standard.

Both cells below use `CohereLabs/tiny-aya-base` (`cohere2`) as the student -- a base checkpoint with no chat template, which AlignTune now handles via a fallback template. Run on a single GPU with 5-10 steps as a smoke test.

**Tokenizer note:** Tiny Aya's tokenizer (vocab 262k) differs from Aya Expanse's (256k). Standard distillation requires the *same* tokenizer for student and teacher, so 15a uses Tiny Aya as its own teacher.

### 15a. Standard distillation (same-tokenizer teacher)

In [ ]:
from aligntune.core.backend_factory import create_distill_trainer

trainer = create_distill_trainer(
    student_model="CohereLabs/tiny-aya-base",
    teacher_model="CohereLabs/tiny-aya-base",  # must share the student's tokenizer
    dataset_name="tatsu-lab/alpaca",
    split="train",
    backend="trl",
    output_dir="./out_distillation",
    batch_size=1,
    max_steps=10,
    learning_rate=5e-5,
    temperature=1.0,
    alpha=0.5,
    loss_type="kl",
    max_seq_length=128,
    max_samples=16,
    use_peft=True,
    lora_r=8,
    lora_alpha=16,
    loggers=["none"],
    seed=42,
    eval_strategy="no",
)

results = trainer.train()
print("Standard distillation training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

### 15b. SDFT (self-distillation fine-tuning)

`batch_size` must be divisible by `num_generations`. `privileged_context_column="input"` points at Alpaca's extra-context field; without it every row is filtered out and training finishes at step 0.

In [ ]:
from aligntune.core.backend_factory import create_distill_trainer

trainer = create_distill_trainer(
    student_model="CohereLabs/tiny-aya-base",
    dataset_name="tatsu-lab/alpaca",
    split="train",
    backend="trl",
    output_dir="./out_sdft",
    batch_size=2,
    max_steps=10,
    learning_rate=5e-5,
    teacher_model_kind="base",
    num_generations=2,
    max_completion_length=32,
    max_seq_length=128,
    max_samples=16,
    privileged_context_column="input",
    use_peft=True,
    lora_r=8,
    lora_alpha=16,
    loggers=["none"],
    seed=42,
    eval_strategy="no",
)

results = trainer.train()
print("SDFT training completed.")
print(results)

# Free GPU memory before the next algorithm cell runs in this kernel
import gc, torch
for _name in ("trainer", "results"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## Done

Ran all 15 algorithms above on the **TRL** backend with `MODEL_NAME`. Each
cell prints its own `trainer.train()` result -- scroll up for per-algorithm
losses/metrics, or re-run just the ones you care about after changing
`MODEL_NAME` in section 0.

To test a different model family, edit the model-selection cell in section 0
and re-run from there down (the setup/install/validation cells above it don't
need to change).